In [39]:
import pandas as pd
import requests
import time
import sqlite3

#### Extração de dados de Licitação

In [40]:
BASE_SEARCH = "https://pncp.gov.br/api/search/"

params = {
    "tipos_documento": "edital",
    "ordenacao": "-data",
    "pagina": 1,
    "tam_pagina": 5,
    "status": "recebendo_proposta",
    "ufs": "PI"
}

resp = requests.get(BASE_SEARCH, params=params)
dados = resp.json()

linhas = []

for compra in dados.get("items", []):

    cnpj = compra.get("orgao_cnpj")
    ano = compra.get("ano")
    sequencial = compra.get("numero_sequencial")

    if not all([cnpj, ano, sequencial]):
        continue

    url = f"https://pncp.gov.br/api/pncp/v1/orgaos/{cnpj}/compras/{ano}/{sequencial}/itens"

    print("URL:", url)

    r = requests.get(url)

    if r.status_code != 200:
        print("Erro:", r.status_code)
        continue

    itens = r.json()

    for item in itens:
        linha = {
            **{f"compra_{k}": v for k, v in compra.items()},
            **{f"item_{k}": v for k, v in item.items()}
        }

        linhas.append(linha)

df = pd.json_normalize(linhas)

URL: https://pncp.gov.br/api/pncp/v1/orgaos/01616855000104/compras/2026/16/itens
URL: https://pncp.gov.br/api/pncp/v1/orgaos/06553630000170/compras/2026/23/itens
URL: https://pncp.gov.br/api/pncp/v1/orgaos/06554356000153/compras/2026/52/itens
URL: https://pncp.gov.br/api/pncp/v1/orgaos/41522376000143/compras/2026/11/itens
URL: https://pncp.gov.br/api/pncp/v1/orgaos/06554158000190/compras/2026/19/itens


#### Subir dados para o Banco SQlite

In [42]:
conn = sqlite3.connect("pncp.db")

df.to_sql("pncp_itens", conn, if_exists="replace", index=False)

14